In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel('test_dataset_anonymized.xlsx')
df.head()  # view first 5 rows

,user_id,is_trial,first_event_date,subscription_renewal_amount,Unnamed: 4,Subs type,Trial period,Subscription period,First payment,Rebill payment
0,21889,1,2022-01-01,0,NaN,trial,7 days,1 month,6.99,29.99
1,21890,1,2019-07-01,0,NaN,wo trial,No trial,3 months,40.00,40.00
2,21891,1,2019-07-01,0,NaN,Subscription is renewed automatically\t\t\t,NaN,NaN,NaN,NaN
3,21892,1,2019-07-01,0,NaN,NaN,NaN,NaN,NaN,NaN
4,21894,1,2019-07-01,0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Let's keep only the columns we need to work with:
df = df[['user_id','is_trial','first_event_date','subscription_renewal_amount']]

# Check the data types:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9426 entries, 0 to 9425
Data columns (total 4 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   user_id                      9426 non-null   int64         
 1   is_trial                     9426 non-null   int64         
 2   first_event_date             9426 non-null   datetime64[us]
 3   subscription_renewal_amount  9426 non-null   int64         
dtypes: datetime64[us](1), int64(3)
memory usage: 294.7 KB


In [ ]:
# check dataframe size:
df.shape

# explore basic stats:
df.describe()


,user_id,is_trial,first_event_date,subscription_renewal_amount
count,9426.000000,9426.000000,9426,9426.000000
mean,23609.311161,0.338320,2019-07-05 04:05:57.479312,5.569489
min,21889.000000,0.000000,2019-07-01 00:00:00,0.000000
25%,22736.000000,0.000000,2019-07-03 00:00:00,0.000000
50%,23572.000000,0.000000,2019-07-06 00:00:00,2.000000
75%,24428.000000,1.000000,2019-07-07 00:00:00,9.000000
max,115217.000000,1.000000,2022-01-01 00:00:00,26.000000
std,1779.225697,0.473163,NaN,6.964244


1: Finding total revenue for trial users: 


In [15]:
# filter trial users
trial_df = df[df['is_trial'] == 0]
trial_df.head()

# calculate revenue per user
trial_df['revenue'] = 6.99 + trial_df['subscription_renewal_amount'] * 29.99

# total revenue
total_revenue_trial = trial_df['revenue'].sum()

print(f'Total revenue for trial users: ${total_revenue_trial:,.2f}')

Total revenue for trial users: $1,617,981.66


2a: Segmenting users by number of renewals:

In [14]:
# Number of users by renewal count:
renewals_by_users = df.groupby('subscription_renewal_amount')['user_id'].count().reset_index().rename(columns={'user_id': 'number_of_users',
                                                                                                    'subscription_renewal_amount': 'number_of_renewals'})

print(renewals_by_users)

    number_of_renewals  number_of_users
0                    0             3188
1                    1              982
2                    2              633
3                    3              498
4                    4              411
5                    5              353
6                    6              321
7                    7              281
8                    8              252
9                    9              236
10                  10              228
11                  11              206
12                  12              190
13                  13              174
14                  14              163
15                  15              147
16                  16              139
17                  17              134
18                  18              126
19                  19              117
20                  20              108
21                  21              102
22                  22               96
23                  23               92


2b: Total Revenue by segments (for Trial only): 

In [17]:
revenue_by_renewals = trial_df.groupby('subscription_renewal_amount')['revenue'].sum().reset_index().rename(columns={'subscription_renewal_amount': 'number_of_renewals', 'revenue': 'revenue_by_renewals'}).sort_values('number_of_renewals')

print(revenue_by_renewals)

    number_of_renewals  revenue_by_renewals
0                    1             36277.38
1                    2             42392.01
2                    3             48286.08
3                    4             52176.45
4                    5             55399.82
5                    6             60004.53
6                    7             60954.52
7                    8             62221.32
8                    9             65348.40
9                   10             69970.92
10                  11             69397.28
11                  12             69705.30
12                  13             69053.64
13                  14             69576.55
14                  15             67155.48
15                  16             67669.37
16                  17             69253.88
17                  18             68898.06
18                  19             67485.60
19                  20             65533.32
20                  21             64951.56
21                  22          

2с: Average LTV by subscription types